# Senior — refine phenotype gates with napariGater

Run these cells with the **`scimap` conda env** as the kernel (top-right kernel picker → select the `scimap` environment, same one you use for Junior's gating).

The napari window opens on this machine's desktop (display `:1`) — keep your remote-desktop view of the box open so you can see and drag the sliders.

Workflow: run cell 1 (load) → cell 2 (opens napari, adjust each marker's threshold + **save gates** in the GUI, then close the window) → cell 3 (persist to the h5ad).

In [1]:
import anndata as ad
import scimap as sm

SEED = '/home/smith6jt/IO60panc2nd/senior_data/singlecell_protein_seeded.h5ad'
adata = ad.read_h5ad(SEED)          # 23.3M cells; big read, RAM is fine
print(adata.shape, '| markers:', len(adata.uns['all_markers']), '| gates seeded:', 'gates' in adata.uns)

Running SCIMAP  2.3.5


/home/smith6jt/miniconda3/envs/scimap/lib/python3.10/site-packages/mpl_scatter_density/__init__.py:4: UserWarning:

pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.



(23280387, 59) | markers: 59 | gates seeded: True


## Seed display contrast from a finer pyramid level

`napariGater`'s auto-contrast samples the **coarsest** pyramid level (`img[-1]`, 32× downsampled here) and sets each channel's display ceiling at the 1–99th percentile of that downsampled sample. Averaging collapses the sparse bright signal, so the ceiling lands **below the real full-res positive-cell intensities** — every positive cell clips to saturated white (unlike QuPath, which uses full-resolution statistics).

This cell recomputes per-channel `low`/`high` from a **finer** level (`STATS_LEVEL=2`) using a 1–99.9th percentile window, and writes them into `adata.uns['image_contrast_settings']['6374']` (overwriting any cached values). Then call `napariGater` with `calculate_contrast=False` so it uses these windows. Tune `PCT_HIGH` up (→99.99) if a marker still looks dim, down (→99) if still saturated; `STATS_LEVEL=3` is faster, `1` is closest to full-res.

In [ ]:
import tifffile as tiff, zarr, dask.array as da, numpy as np

QPTIFF      = '/home/smith6jt/IO60panc2nd/Images/6374_Scan1.er.qptiff'
IMAGE_ID    = '6374'
STATS_LEVEL = 2          # 0=full-res(slow) … 5=scimap default(too downsampled). 2 is a good tradeoff
PCT_LOW, PCT_HIGH = 1.0, 99.9

z   = zarr.open(tiff.TiffFile(QPTIFF, is_ome=False).aszarr(), mode='r')
arr = da.from_zarr(z[STATS_LEVEL])           # (n_channels, H, W)
markers = list(adata.uns['all_markers'])

settings = {}
for i, m in enumerate(markers):
    s  = np.asarray(arr[i]).ravel()          # one channel of level 2 (~13 GB total over 59 ch; a few min)
    s  = s[np.isfinite(s)]
    lo = float(np.percentile(s, PCT_LOW))
    hi = float(np.percentile(s, PCT_HIGH))
    settings[m] = {'low': lo, 'high': max(hi, lo + 1)}

adata.uns.setdefault('image_contrast_settings', {})[IMAGE_ID] = settings   # overwrite the bad L5 cache
print(f'seeded contrast for {IMAGE_ID} from level {STATS_LEVEL} ({len(settings)} channels)')

In [ ]:
# Opens napari on the 6374 (ND) image. Streams lazily (dask pyramid) — the 120 GB qptiff is NOT loaded into RAM.
# Gates are GLOBAL: you tune them on 6374 but they apply to all 15 donors. Spot-check a T1D donor (subset='6593') too.
sm.pl.napariGater(
    '/home/smith6jt/IO60panc2nd/Images/6374_Scan1.er.qptiff',
    adata,
    subset='6374',
    layer='raw',
    log=True,                 # gates in log1p scale (sm.pp.rescale expects this)
    centroid_units='microns', # Senior centroids are in µm (QuPath)
    flip_y=False,             # matches Junior's napariGater call
    calculate_contrast=False, # use the contrast windows seeded in the cell above
    # pixel_size_um=0.2505,   # uncomment if the cell overlay is misaligned with the image
)

In [ ]:
# After adjusting + saving gates in the napari GUI and closing the window, persist them:
adata.write_h5ad(SEED)
print('saved refined gates ->', SEED)